### Scaled Dot-Product Attention Implementation

The Scaled Dot-Product Attention mathematically computes attention scores using Queries ($Q$), Keys ($K$), and Values ($V$):

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Where:
* $d_k$ is the dimensionality of the key vectors (used for scaling to prevent extremely small gradients during training).
* $\text{softmax}$ is applied along the last axis to normalize the scores into probabilities.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super(ScaledDotProductAttention, self).__init__()

    def forward(self, Q, K, V, mask=None):
        # Q, K, V shape: [batch_size, num_heads, sequence_length, d_k]
        d_k = Q.size(-1)

        # 1. Compute attention scores (matmul of Q and K transposed)
        # Scores shape: [batch_size, num_heads, sequence_length, sequence_length]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

        # 2. Apply mask (optional, e.g., for autoregressive masking in decoders)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        # 3. Apply softmax to get attention weights
        attention_weights = F.softmax(scores, dim=-1)

        # 4. Multiply by values V
        output = torch.matmul(attention_weights, V)

        return output, attention_weights

### Let's verify with a simple demo
We will define dummy inputs for $Q$, $K$, and $V$ and pass them through our attention layer.

In [2]:
# Batch size: 2, Heads: 1, Sequence length: 3, Feature dimension (d_k): 4
batch_size = 2
num_heads = 1
seq_len = 3
d_k = 4

torch.manual_seed(42)
Q = torch.randn(batch_size, num_heads, seq_len, d_k)
K = torch.randn(batch_size, num_heads, seq_len, d_k)
V = torch.randn(batch_size, num_heads, seq_len, d_k)

attention_layer = ScaledDotProductAttention()
output, weights = attention_layer(Q, K, V)

print("Output Shape:", output.shape)
print("Attention Weights Shape:", weights.shape)
print("\nAttention Weights (First Batch):")
print(weights[0, 0])

Output Shape: torch.Size([2, 1, 3, 4])
Attention Weights Shape: torch.Size([2, 1, 3, 3])

Attention Weights (First Batch):
tensor([[0.2746, 0.2220, 0.5034],
        [0.4423, 0.3890, 0.1687],
        [0.2303, 0.5218, 0.2479]])
